## Introduction

This notebook provides a simple example of how to use the **kernelized Taylor diagram** for
visualizing similarities in data. We generate some random data and transform it with
progressively more complex transformations, then visualize how the similarity to the
original data diminishes.

The implementation lives in the `ktd` package in this repository, which is based on the paper:

> Wickstrøm, K., Johnson, J. E., Løkse, S., Camps-Valls, G., Mikalsen, K. Ø., Kampffmeyer, M., & Jenssen, R. (2022).
> *The Kernelized Taylor Diagram*. arXiv:2205.08864. https://arxiv.org/abs/2205.08864

The kernel similarity computation (`HSIC`) is adapted from the
[PySim](https://github.com/jejjohnson/pysim) package.

In [1]:
import matplotlib.pyplot as plt
import numpy as np

from ktd import HSIC, TaylorDiagram

np.random.seed(1)

N = 500
x = np.random.uniform(0, np.pi, size=(N, 1))

## Calculating similarities

The following block calculates similarities between `x` and different transformations of `x`
using an RBF kernel. For each transformation, we compute:

- the **kernel alignment score** (a CKA score between the reference kernel and the transformed kernel),
  which becomes the *angle* of the point, and
- the **logarithm of the Frobenius norm** of the transformed kernel matrix,
  which becomes the *radius* of the point.

In [2]:
kernel = "rbf"


def alignment_and_norm(x, z):
    clf_kernel = HSIC(center=True, kernel=kernel, gamma_X=1.0, gamma_Y=1.0)
    clf_kernel.fit(x, z)
    return clf_kernel.score(normalize=True), np.log(clf_kernel.K_y_norm)


# reference transformation
clf_kernel = HSIC(center=True, kernel=kernel, gamma_X=1.0, gamma_Y=1.0)
clf_kernel.fit(x, x)
Fx = np.log(clf_kernel.K_x_norm)

func_list = [lambda x: x, lambda x: np.sin(x), lambda x: np.sin(x**2)]
points = np.zeros((len(func_list), 2))

for idx, func in enumerate(func_list):
    z = func(x) + np.random.normal(0, 0.1, size=x.shape)
    points[idx] = alignment_and_norm(x, z)

points

array([[0.98041488, 5.16708564],
       [0.41632708, 4.35467576],
       [0.21811883, 4.88112325]])

## Plotting the kernelized Taylor diagram

This block plots the actual diagram. The figure shows how similarity diminishes as the
transformation becomes more complex, and saves the result to `figures/example.png`.

In [3]:
from pathlib import Path

def find_repo_root() -> Path:
    """Locate the repository root (works regardless of the kernel's CWD)."""
    cwd = Path.cwd()
    for cand in [cwd, *cwd.parents]:
        if (cand / "pyproject.toml").exists():
            return cand
    return cwd

fig = plt.figure(figsize=(10, 10))

taylor_fig = TaylorDiagram(
    ref_point=Fx,
    fig=fig,
    subplot=111,
    extend_angle=False,
    ref_range=(0, 30),
    angle_label="Kernel alignment score",
    var_label="Logarithm of Frobenius norm",
)

# reference point
taylor_fig.add_reference_point(Fx, color="black", marker=".", markersize=20, label="ref")

# reference line
taylor_fig.add_reference_line(Fx, color="black", linestyle="--", label="_")

# grid and contours
taylor_fig.add_grid()
taylor_fig.add_contours(Fx, levels=3, colors="gray")
taylor_fig.polar_axes.clabel(taylor_fig.contours, inline=1, fontsize=20, fmt="%.1f")

# sample points
mark_list = [">", "p", "<", "s"]
col_list = ["green", "blue", "orange", "black"]
lab_list = ["f(x)=x", "f(x)=sin(x)", "f(x)=sin(x\u00b2)"]

for i in range(len(func_list)):
    taylor_fig.add_scatter(
        [points[i, 1]],
        [points[i, 0]],
        c=col_list[i],
        s=200,
        marker=mark_list[i],
        label=lab_list[i],
        zorder=3,
        edgecolors="black",
        alpha=0.75,
    )

taylor_fig.graph_axes.legend(ncol=4, bbox_to_anchor=(1.0, 1.125), prop={"size": 8})

figures_dir = find_repo_root() / "figures"
figures_dir.mkdir(exist_ok=True)
plt.savefig(figures_dir / "example.png", bbox_inches="tight", dpi=150)
plt.show()

/var/folders/kw/6z9hvl4977jgcbj5ty886xg00000gn/T/ipykernel_61838/1110784442.py:57: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Citation

If you use this code, please cite the paper:

```bibtex
@article{wickstrom2022kernelized,
  title={The Kernelized Taylor Diagram},
  author={Wickstr{\o}m, Kristoffer and Johnson, J. Emmanuel and L{\o}kse, Sigurd and Camps-Valls, Gustau and Mikalsen, Karl {\O}yvind and Kampffmeyer, Michael and Jenssen, Robert},
  journal={arXiv preprint arXiv:2205.08864},
  year={2022}
}
```